In [ ]:
# Import necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from zipfile import ZipFile

from sklearn.preprocessing import power_transform, OneHotEncoder, LabelEncoder, OrdinalEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

import os
import joblib

## Load Clean Data

In [29]:
path = r"C:\Users\ncc892\Desktop\kaggle_competition\playground-series-s5e11.zip"

with ZipFile(path , 'r') as zippath:
    zippath.printdir()


with ZipFile(path, 'r') as zipref:
    with zipref.open('train.csv') as file:
        df = pd.read_csv(file)
    
    with zipref.open('test.csv') as data:
        test_data = pd.read_csv(data)


File Name                                             Modified             Size
sample_submission.csv                          2025-10-28 23:08:48      2291139
test.csv                                       2025-10-28 23:08:48     23021430
train.csv                                      2025-10-28 23:08:50     55988519


In [ ]:
# data = pd.read_csv('eda_cleaned_train_data.csv')

# df = data.copy()


In [30]:
X = df.drop(columns=['loan_paid_back', 'id'])
y = df['loan_paid_back']


In [31]:
# Trim X
numerical_cols = X.select_dtypes(include='number').columns

categorical_cols = X.select_dtypes(include='object').columns


## Handle Skewness

In [ ]:

# df[numerical_cols] = power_transform(df[numerical_cols], method = 'yeo-johnson')
# print(numerical_cols)

Index(['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount',
       'interest_rate'],
      dtype='object')


## Encode Categorical Data

In [35]:
# Create your list of categorical data for one-hot and ordinal
encode_list = ['gender', 'marital_status', 'employment_status', 'loan_purpose']
ordinal_columns = ['education_level', 'grade_subgrade']

# Mapped Ordinal Data
education_order = ["Other", "High School", "Bachelor's", "Master's", "PhD"]
grade_order = ['A1', 'A2', 'A3', 'A4', 'A5', 'B1', 'B2', 'B3', 'B4', 'B5', 'C1', 'C2', 'C3', 'C4', 'C5', 'D1', 'D2', 'D3', 'D4', 'D5', 'E1', 'E2', 'E3', 'E4', 'E5', 'F1', 'F2', 'F3', 'F4', 'F5']


# Initialize ordinal encoder
ordinal = OrdinalEncoder(categories=[education_order, grade_order], handle_unknown='use_encoded_value', unknown_value=-1)
# Initialize one-hot encoder
one_hot = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)


# Call the column transformer
column_transformer = ColumnTransformer(transformers=
                                       [
                                        ("ordinal_enc", ordinal, ordinal_columns),
                                        ('one_hot_enc', one_hot, encode_list),
                                        # include the numeric columns(though no operation will be performed on them)
                                        ("num", PowerTransformer(), numerical_cols)
                                        ]
                                       )

# Fit the column transformer on the training data (X)
column_transformer.fit(X)

# Save the column transformer 
os.makedirs("artifacts", exist_ok=True)
joblib.dump(column_transformer, "artifacts/column_preprocessor.pkl")

['artifacts/column_preprocessor.pkl']

# Model Building

In [ ]:
# Second level splitting
X_train, x_val, y_train, y_val = train_test_split()

# define the models to be used
models = {
    "Linear Regression": LinearRegression(),
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier()
}

# Loop through
for name,  model in enumerate(models):
    # print model name
    print(f'Running {name} ')
    model.fit(X)
    print(f'{}')
    
